Project Setup

In [ ]:
import pandas as pd
import numpy as np

Upload XER File

In [ ]:
from google.colab import files

uploaded = files.upload()
xer_file = list(uploaded.keys())[0]

print("Uploaded file:", xer_file)

Saving Hotel Project.xer to Hotel Project (5).xer
Uploaded file: Hotel Project (5).xer


XER table reader

In [ ]:
def read_xer_table(file_path, table_name):
    rows = []
    columns = None
    inside_table = False

    with open(file_path, "r", encoding="latin-1", errors="ignore") as file:
        for line in file:
            line = line.strip("\n")
            parts = line.split("\t")

            if len(parts) > 1 and parts[0] == "%T" and parts[1] == table_name:
                inside_table = True
                continue

            if inside_table and parts[0] == "%F":
                columns = parts[1:]
                continue

            if inside_table and parts[0] == "%R":
                rows.append(parts[1:])
                continue

            if inside_table and parts[0] == "%T":
                break

    return pd.DataFrame(rows, columns=columns)

Load Primavera tables

In [ ]:
projects = read_xer_table(xer_file, "PROJECT")
wbs = read_xer_table(xer_file, "PROJWBS")
tasks = read_xer_table(xer_file, "TASK")
rels = read_xer_table(xer_file, "TASKPRED")

print("Projects:", projects.shape)
print("WBS:", wbs.shape)
print("Activities:", tasks.shape)
print("Relationships:", rels.shape)

Projects: (4, 71)
WBS: (1025, 26)
Activities: (4236, 61)
Relationships: (7586, 10)


Check available projects

In [ ]:
projects[["proj_id", "proj_short_name"]]

,proj_id,proj_short_name
0,2661,DS
1,2663,OP
2,2665,CR
3,2666,HBTF-2


Select HBTF-2 project

In [ ]:
TARGET_PROJECT_ID = "2666"   # HBTF-2

tasks_project = tasks[tasks["proj_id"] == TARGET_PROJECT_ID].copy()
wbs_project = wbs[wbs["proj_id"] == TARGET_PROJECT_ID].copy()

project_task_ids = set(tasks_project["task_id"])

rels_project = rels[
    rels["task_id"].isin(project_task_ids) |
    rels["pred_task_id"].isin(project_task_ids)
].copy()

print("Selected Project Activities:", tasks_project.shape)
print("Selected Project WBS:", wbs_project.shape)
print("Selected Project Relationships:", rels_project.shape)

Selected Project Activities: (4217, 61)
Selected Project WBS: (964, 26)
Selected Project Relationships: (7586, 10)


Clean numeric fields

In [ ]:
tasks_project["total_float_days"] = pd.to_numeric(
    tasks_project["total_float_hr_cnt"], errors="coerce"
) / 8

tasks_project["remaining_days"] = pd.to_numeric(
    tasks_project["remain_drtn_hr_cnt"], errors="coerce"
) / 8

tasks_project["duration_days"] = pd.to_numeric(
    tasks_project["target_drtn_hr_cnt"], errors="coerce"
) / 8

Add successor count

In [ ]:
successor_count = rels_project.groupby("pred_task_id").size().reset_index(
    name="successor_count"
)

tasks_project = tasks_project.merge(
    successor_count,
    left_on="task_id",
    right_on="pred_task_id",
    how="left"
)

tasks_project["successor_count"] = tasks_project["successor_count"].fillna(0)

Merge activities with WBS

In [ ]:
data = tasks_project.merge(
    wbs_project[["wbs_id", "parent_wbs_id", "wbs_short_name", "wbs_name"]],
    on="wbs_id",
    how="left"
)

data.head()

,task_id,proj_id,wbs_id,clndr_id,est_wt,phys_complete_pct,rev_fdbk_flag,lock_plan_flag,auto_compute_act_flag,complete_pct_type,...,create_user,update_user,total_float_days,remaining_days,duration_days,pred_task_id,successor_count,parent_wbs_id,wbs_short_name,wbs_name
0,143649,2666,25860,3195,1,0,N,N,N,CP_Drtn,...,NotPrmUser,admin,2.0,13.0,13.0,143649,3.0,25730,1,Zone # 1
1,143650,2666,25880,3195,1,0,N,N,N,CP_Drtn,...,NotPrmUser,admin,2.0,11.0,11.0,143650,2.0,25860,1,Foundations - Part 1
2,143651,2666,25702,3195,1,0,N,N,N,CP_Drtn,...,NotPrmUser,admin,37.0,12.0,12.0,143651,3.0,25700,2,Mobilzation
3,143652,2666,25702,3195,1,0,N,N,N,CP_Drtn,...,NotPrmUser,admin,30.0,20.0,20.0,143652,2.0,25700,2,Mobilzation
4,143653,2666,25702,3195,1,0,N,N,N,CP_Drtn,...,NotPrmUser,admin,14.0,19.0,19.0,143653,2.0,25700,2,Mobilzation


. Project health summary

In [ ]:
project_health = {
    "Total Activities": len(data),
    "Critical / Zero Float Activities": (data["total_float_days"] <= 0).sum(),
    "Average Float": data["total_float_days"].mean(),
    "Total Remaining Days": data["remaining_days"].sum(),
    "Total Relationships": len(rels_project),
    "Total WBS Packages": data["wbs_id"].nunique()
}

project_health

{'Total Activities': 4217,
 'Critical / Zero Float Activities': np.int64(197),
 'Average Float': np.float64(65.22438937633389),
 'Total Remaining Days': np.float64(53547.0),
 'Total Relationships': 7586,
 'Total WBS Packages': 755}

Package exposure analysis

In [ ]:
package_exposure = data.groupby(["wbs_short_name", "wbs_name"]).agg(
    total_activities=("task_id", "count"),
    critical_activities=("total_float_days", lambda x: (x <= 0).sum()),
    avg_float=("total_float_days", "mean"),
    total_remaining_days=("remaining_days", "sum"),
    total_successors=("successor_count", "sum")
).reset_index()

package_exposure["critical_density"] = (
    package_exposure["critical_activities"] /
    package_exposure["total_activities"]
) * 100

Package exposure score

In [ ]:
package_exposure["baseline_exposure_score"] = (
    package_exposure["critical_density"] * 0.40
    + package_exposure["critical_activities"] * 0.30
    + package_exposure["total_successors"] * 0.20
    + package_exposure["total_remaining_days"] * 0.01
)

package_exposure.sort_values(
    "baseline_exposure_score",
    ascending=False
).head(20)

,wbs_short_name,wbs_name,total_activities,critical_activities,avg_float,total_remaining_days,total_successors,critical_density,baseline_exposure_score
1,020,Electrical Works,567,26,84.626764,4159.00,1033.0,4.585538,257.824215
125,WL,Wall Finishes,444,23,83.648086,4526.50,968.0,5.180180,247.837072
119,FL,Floor Finishes,423,19,80.545508,3891.00,661.0,4.491726,178.606690
110,CL,Ceiling Finishes,301,21,65.732558,3058.25,659.0,6.976744,171.473198
5,1,Air Conditioning,260,13,27.930288,3153.00,373.0,5.000000,112.030000
44,2,Steel Structure Works,144,71,19.965278,419.00,292.0,49.305556,103.612222
18,1,Retaining Walls,192,0,36.519531,1358.00,416.0,0.000000,96.780000
121,MS,Mecellanuos,183,0,60.882514,1952.00,348.0,0.000000,89.120000
92,7,Drainage,156,0,89.036058,3120.00,262.0,0.000000,83.600000
122,MS,Miscellaneous,140,0,72.294643,1507.00,311.0,0.000000,77.270000


Executive watchlist

In [ ]:
executive_watchlist = package_exposure.sort_values(
    "baseline_exposure_score",
    ascending=False
).head(10)

executive_watchlist[
    [
        "wbs_short_name",
        "wbs_name",
        "total_activities",
        "critical_activities",
        "critical_density",
        "total_remaining_days",
        "total_successors",
        "baseline_exposure_score"
    ]
]

,wbs_short_name,wbs_name,total_activities,critical_activities,critical_density,total_remaining_days,total_successors,baseline_exposure_score
1,020,Electrical Works,567,26,4.585538,4159.00,1033.0,257.824215
125,WL,Wall Finishes,444,23,5.180180,4526.50,968.0,247.837072
119,FL,Floor Finishes,423,19,4.491726,3891.00,661.0,178.606690
110,CL,Ceiling Finishes,301,21,6.976744,3058.25,659.0,171.473198
5,1,Air Conditioning,260,13,5.000000,3153.00,373.0,112.030000
44,2,Steel Structure Works,144,71,49.305556,419.00,292.0,103.612222
18,1,Retaining Walls,192,0,0.000000,1358.00,416.0,96.780000
121,MS,Mecellanuos,183,0,0.000000,1952.00,348.0,89.120000
92,7,Drainage,156,0,0.000000,3120.00,262.0,83.600000
122,MS,Miscellaneous,140,0,0.000000,1507.00,311.0,77.270000


Activity Baseline Risk Engine

In [ ]:
   # -------------------------------
# Activity Baseline Risk Engine
# PMO Logic:
# "If this activity slips before execution starts,
# how much damage can it cause to the project?"
# -------------------------------

# 1. Add package criticality back to each activity
package_metrics = package_exposure[
    [
        "wbs_short_name",
        "wbs_name",
        "critical_density",
        "baseline_exposure_score"
    ]
].copy()

activity_data = data.merge(
    package_metrics,
    on=["wbs_short_name", "wbs_name"],
    how="left"
)

# 2. Float Risk Score: 0-100
def float_risk_score(float_days):
    if pd.isna(float_days):
        return 0
    if float_days <= 0:
        return 100
    elif float_days <= 5:
        return 80
    elif float_days <= 15:
        return 50
    else:
        return 20

# 3. Dependency Risk Score: 0-100
def dependency_risk_score(successors):
    if pd.isna(successors):
        return 0
    if successors > 20:
        return 100
    elif successors >= 10:
        return 80
    elif successors >= 5:
        return 50
    else:
        return 20

# 4. Duration Risk Score: 0-100
def duration_risk_score(duration):
    if pd.isna(duration):
        return 0
    if duration > 30:
        return 100
    elif duration >= 15:
        return 70
    elif duration > 0:
        return 30
    else:
        return 0

# 5. Package Risk Score: 0-100
# Based on critical density of the package
def package_risk_score(critical_density):
    if pd.isna(critical_density):
        return 0
    if critical_density >= 40:
        return 100
    elif critical_density >= 20:
        return 75
    elif critical_density >= 10:
        return 50
    elif critical_density > 0:
        return 25
    else:
        return 0

# 6. Milestone / Gateway Risk Score
# Detect activities that act like gateways or handover points
def gateway_risk_score(row):
    name = str(row["task_name"]).lower()
    successors = row["successor_count"]

    gateway_keywords = [
        "milestone",
        "handover",
        "approval",
        "inspection",
        "testing",
        "commissioning",
        "complete",
        "completion",
        "permit",
        "authority"
    ]

    keyword_match = any(word in name for word in gateway_keywords)

    if keyword_match and successors >= 5:
        return 100
    elif keyword_match:
        return 75
    elif successors >= 15:
        return 60
    else:
        return 20

# 7. Apply individual scores
activity_data["float_risk"] = activity_data["total_float_days"].apply(float_risk_score)
activity_data["dependency_risk"] = activity_data["successor_count"].apply(dependency_risk_score)
activity_data["duration_risk"] = activity_data["remaining_days"].apply(duration_risk_score)
activity_data["package_risk"] = activity_data["critical_density"].apply(package_risk_score)
activity_data["gateway_risk"] = activity_data.apply(gateway_risk_score, axis=1)

# 8. Final weighted PMO Activity Risk Score
activity_data["activity_risk_score"] = (
    activity_data["float_risk"] * 0.40
    + activity_data["dependency_risk"] * 0.25
    + activity_data["duration_risk"] * 0.15
    + activity_data["package_risk"] * 0.10
    + activity_data["gateway_risk"] * 0.10
)

# 9. Risk category
def risk_category(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

activity_data["risk_category"] = activity_data["activity_risk_score"].apply(risk_category)

# 10. Generate risk explanation
def risk_reason(row):
    reasons = []

    if row["float_risk"] >= 80:
        reasons.append("Low/zero float")
    if row["dependency_risk"] >= 80:
        reasons.append("High successor impact")
    if row["duration_risk"] >= 70:
        reasons.append("Long duration exposure")
    if row["package_risk"] >= 75:
        reasons.append("Located in high-criticality package")
    if row["gateway_risk"] >= 75:
        reasons.append("Gateway/milestone-type activity")

    if not reasons:
        reasons.append("Limited baseline exposure")

    return " + ".join(reasons)

activity_data["risk_reason"] = activity_data.apply(risk_reason, axis=1)

# 11. Final Activity Risk Report
activity_risk_report = activity_data[
    [
        "task_code",
        "task_name",
        "wbs_name",
        "total_float_days",
        "remaining_days",
        "successor_count",
        "critical_density",
        "float_risk",
        "dependency_risk",
        "duration_risk",
        "package_risk",
        "gateway_risk",
        "activity_risk_score",
        "risk_category",
        "risk_reason"
    ]
].sort_values(
    "activity_risk_score",
    ascending=False
)

activity_risk_report.head(20)

,task_code,task_name,wbs_name,total_float_days,remaining_days,successor_count,critical_density,float_risk,dependency_risk,duration_risk,package_risk,gateway_risk,activity_risk_score,risk_category,risk_reason
572,H0000,Start Date,Project Milestone,0.0,1.0,32.0,100.000000,100,100,30,100,60,85.5,Critical,Low/zero float + High successor impact + Locat...
782,HCESCA000,s/c approval: MEP subcontractor,Approval,0.0,14.0,12.0,50.000000,100,80,30,100,100,84.5,Critical,Low/zero float + High successor impact + Locat...
785,HESDSSS0000,SD submission: Steel Strucutre shopdrawings,Steel Strucuture,0.0,100.0,6.0,100.000000,100,50,100,100,20,79.5,High,Low/zero float + Long duration exposure + Loca...
786,HESDASS0000,SD approval: Steel Strucutre shopdrawings,Steel Strucuture,0.0,100.0,1.0,100.000000,100,20,100,100,75,77.5,High,Low/zero float + Long duration exposure + Loca...
855,HEPSAME1160,Diffusers,MEP Materials,0.0,240.0,6.0,13.043478,100,50,100,50,20,74.5,High,Low/zero float + Long duration exposure
858,HEPSAME1300,CableTrays,MEP Materials,0.0,240.0,6.0,13.043478,100,50,100,50,20,74.5,High,Low/zero float + Long duration exposure
3706,HCSWAFAZ000,Construct of left over areas (crane # 3),Casting and Finishing of Left over slabs,0.0,49.0,2.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
3708,HCFWAFAZ000,Finish Left over areas (Crane #3),Casting and Finishing of Left over slabs,0.0,142.0,2.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
3709,HCFWAFAZ010,Finish Left over areas (Crane #2),Casting and Finishing of Left over slabs,0.0,81.0,1.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
644,HCSW4FZ1090,Casting concrete for slab topping,Steel Structure Works,0.0,1.0,5.0,49.305556,100,50,30,100,20,69.0,High,Low/zero float + Located in high-criticality p...


Validation Cell 1

In [ ]:
activity_risk_report.head(20)[
[
    "task_code",
    "task_name",
    "total_float_days",
    "successor_count",
    "remaining_days",
    "activity_risk_score"
]
]

,task_code,task_name,total_float_days,successor_count,remaining_days,activity_risk_score
572,H0000,Start Date,0.0,32.0,1.0,85.5
782,HCESCA000,s/c approval: MEP subcontractor,0.0,12.0,14.0,84.5
785,HESDSSS0000,SD submission: Steel Strucutre shopdrawings,0.0,6.0,100.0,79.5
786,HESDASS0000,SD approval: Steel Strucutre shopdrawings,0.0,1.0,100.0,77.5
855,HEPSAME1160,Diffusers,0.0,6.0,240.0,74.5
858,HEPSAME1300,CableTrays,0.0,6.0,240.0,74.5
3706,HCSWAFAZ000,Construct of left over areas (crane # 3),0.0,2.0,49.0,72.0
3708,HCFWAFAZ000,Finish Left over areas (Crane #3),0.0,2.0,142.0,72.0
3709,HCFWAFAZ010,Finish Left over areas (Crane #2),0.0,1.0,81.0,72.0
644,HCSW4FZ1090,Casting concrete for slab topping,0.0,5.0,1.0,69.0


In [ ]:
activity_data[
    activity_data["total_float_days"] <= 0
].shape

(197, 79)

In [ ]:
activity_data[
    activity_data["activity_risk_score"] >= 70
].shape

(9, 79)

In [ ]:
activity_data[
[
    "task_code",
    "task_name",
    "float_risk",
    "dependency_risk",
    "duration_risk",
    "package_risk",
    "gateway_risk",
    "activity_risk_score"
]
].sort_values(
    "activity_risk_score",
    ascending=False
).head(20)

,task_code,task_name,float_risk,dependency_risk,duration_risk,package_risk,gateway_risk,activity_risk_score
572,H0000,Start Date,100,100,30,100,60,85.5
782,HCESCA000,s/c approval: MEP subcontractor,100,80,30,100,100,84.5
785,HESDSSS0000,SD submission: Steel Strucutre shopdrawings,100,50,100,100,20,79.5
786,HESDASS0000,SD approval: Steel Strucutre shopdrawings,100,20,100,100,75,77.5
855,HEPSAME1160,Diffusers,100,50,100,50,20,74.5
858,HEPSAME1300,CableTrays,100,50,100,50,20,74.5
3706,HCSWAFAZ000,Construct of left over areas (crane # 3),100,20,100,100,20,72.0
3708,HCFWAFAZ000,Finish Left over areas (Crane #3),100,20,100,100,20,72.0
3709,HCFWAFAZ010,Finish Left over areas (Crane #2),100,20,100,100,20,72.0
644,HCSW4FZ1090,Casting concrete for slab topping,100,50,30,100,20,69.0


In [ ]:
activity_risk_report_no_milestones = activity_risk_report[
    ~activity_risk_report["task_name"].str.lower().str.contains(
        "start date",
        na=False
    )
]

activity_risk_report_no_milestones.head(20)

,task_code,task_name,wbs_name,total_float_days,remaining_days,successor_count,critical_density,float_risk,dependency_risk,duration_risk,package_risk,gateway_risk,activity_risk_score,risk_category,risk_reason
782,HCESCA000,s/c approval: MEP subcontractor,Approval,0.0,14.0,12.0,50.000000,100,80,30,100,100,84.5,Critical,Low/zero float + High successor impact + Locat...
785,HESDSSS0000,SD submission: Steel Strucutre shopdrawings,Steel Strucuture,0.0,100.0,6.0,100.000000,100,50,100,100,20,79.5,High,Low/zero float + Long duration exposure + Loca...
786,HESDASS0000,SD approval: Steel Strucutre shopdrawings,Steel Strucuture,0.0,100.0,1.0,100.000000,100,20,100,100,75,77.5,High,Low/zero float + Long duration exposure + Loca...
855,HEPSAME1160,Diffusers,MEP Materials,0.0,240.0,6.0,13.043478,100,50,100,50,20,74.5,High,Low/zero float + Long duration exposure
858,HEPSAME1300,CableTrays,MEP Materials,0.0,240.0,6.0,13.043478,100,50,100,50,20,74.5,High,Low/zero float + Long duration exposure
3706,HCSWAFAZ000,Construct of left over areas (crane # 3),Casting and Finishing of Left over slabs,0.0,49.0,2.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
3708,HCFWAFAZ000,Finish Left over areas (Crane #3),Casting and Finishing of Left over slabs,0.0,142.0,2.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
3709,HCFWAFAZ010,Finish Left over areas (Crane #2),Casting and Finishing of Left over slabs,0.0,81.0,1.0,75.000000,100,20,100,100,20,72.0,High,Low/zero float + Long duration exposure + Loca...
644,HCSW4FZ1090,Casting concrete for slab topping,Steel Structure Works,0.0,1.0,5.0,49.305556,100,50,30,100,20,69.0,High,Low/zero float + Located in high-criticality p...
631,HCSW3FZ2090,Casting concrete for slab topping,Steel Structure Works,0.0,1.0,5.0,49.305556,100,50,30,100,20,69.0,High,Low/zero float + Located in high-criticality p...
